In [75]:
import pandas as pd

df_stage_1 = pd.read_csv('csv_input/county_layer.csv')
df_stage_2 = pd.read_csv('csv_input/stage2_synced_polygons_updated.csv')
df_stage_1 = df_stage_1[['County', 'Shape_ID', 'geometry']]
df_stage_2 = df_stage_2[['County', 'Shape_ID', 'geometry']]

In [76]:
import geopandas as gpd
from shapely.ops import unary_union
from shapely import wkt as shapely_wkt
import re

county_order = df_stage_1['County'].unique().tolist()

# Convert to GeoDataFrames
gdf_stage_1 = gpd.GeoDataFrame(df_stage_1, geometry=gpd.GeoSeries.from_wkt(df_stage_1['geometry']))

def fix_wkt(wkt_str):
    """Try to close unclosed rings in a WKT POLYGON string."""
    try:
        return shapely_wkt.loads(wkt_str)
    except Exception:
        # Close each ring by appending its first coordinate to the end
        def close_ring(ring_str):
            coords = ring_str.strip().split(',')
            if coords[0].strip() != coords[-1].strip():
                coords.append(coords[0])
            return ', '.join(coords)
        fixed = re.sub(r'\(([^()]+)\)', lambda m: '(' + close_ring(m.group(1)) + ')', wkt_str)
        try:
            return shapely_wkt.loads(fixed)
        except Exception:
            return None

fixed_geoms = df_stage_2['geometry'].apply(fix_wkt)
n_fixed = fixed_geoms.isna().sum()
n_repaired = (gpd.GeoSeries.from_wkt(df_stage_2['geometry'], on_invalid='warn').isna() & fixed_geoms.notna()).sum()

if n_repaired:
    print(f"{n_repaired} invalid geometry(s) auto-repaired by closing ring.")
if n_fixed:
    print(f"{n_fixed} geometry(s) could not be repaired and will be dropped.")
    print(df_stage_2[fixed_geoms.isna()][['County', 'Shape_ID']])

gdf_stage_2 = gpd.GeoDataFrame(df_stage_2, geometry=fixed_geoms)
gdf_stage_2 = gdf_stage_2[gdf_stage_2.geometry.notna()].reset_index(drop=True)

# --- Hole detection & fix ---
UNITS_PER_KM = 8.01

for county in county_order:
    union_s1 = unary_union(gdf_stage_1[gdf_stage_1['County'] == county].geometry)
    union_s2 = unary_union(gdf_stage_2[gdf_stage_2['County'] == county].geometry)
    hole = union_s1.difference(union_s2)
    if not hole.is_empty:
        holes = list(hole.geoms) if hole.geom_type == 'MultiPolygon' else [hole]
        for i, h in enumerate(holes):
            shape_id = f"{county}_{999 - i}"
            print(f"Hole found in {county} [{shape_id}] — {h.area / (UNITS_PER_KM ** 2):.4f} km²")
            new_row = gpd.GeoDataFrame({'County': [county], 'Shape_ID': [shape_id], 'geometry': [h]}, crs=gdf_stage_2.crs)
            gdf_stage_2 = pd.concat([gdf_stage_2, new_row], ignore_index=True)

# Compute centroids
gdf_stage_1['centroid'] = gdf_stage_1.geometry.centroid
gdf_stage_2['centroid'] = gdf_stage_2.geometry.centroid

# Compute area in km²
gdf_stage_1['Area'] = (gdf_stage_1.geometry.area / (UNITS_PER_KM ** 2)).round(4)
gdf_stage_2['Area'] = (gdf_stage_2.geometry.area / (UNITS_PER_KM ** 2)).round(4)

# Check county area sums match (tolerance ±0.001 km²)
TOLERANCE = 0.001
area_s1 = gdf_stage_1.groupby('County')['Area'].sum().round(4)
area_s2 = gdf_stage_2.groupby('County')['Area'].sum().round(4)

diff = (area_s1 - area_s2).dropna()
diff = diff[diff.abs() > TOLERANCE]

if diff.empty:
    print("All county area sums match.")
else:
    print("Mismatched counties:")
    for county, delta in diff.items():
        print(f"  {county}: stage_1={area_s1[county]:.4f}, stage_2={area_s2[county]:.4f}, diff={delta:.4f} km²")

gdf_stage_2.sort_values('Area').reset_index(drop=True)[['County', 'Shape_ID', 'Area']]

1 invalid geometry(s) auto-repaired by closing ring.
All county area sums match.


C:\Users\ngman\AppData\Roaming\Python\Python313\site-packages\shapely\io.py:305: Warning: Invalid WKT: geometry is returned as None. IllegalArgumentException: Points of LinearRing do not form a closed linestring
  return lib.from_wkt(geometry, invalid_handler, **kwargs)


,County,Shape_ID,Area
0,E12,E12_074,0.0289
1,WL1,WL1_011,0.0351
2,NC5,NC5_024,0.0374
3,E13,E13_046,0.0401
4,E3,E3_028,0.0407
...,...,...,...
1704,WL2,WL2_021,69.7766
1705,W3,W3_06,72.0262
1706,WL1,WL1_007,75.6871
1707,WL2,WL2_056,83.9868


In [77]:
import numpy as np

# Compute each county's centroid from stage_1 (union all polygons → centroid)
county_centroids = (
    gdf_stage_1.groupby('County')['geometry']
    .apply(lambda geoms: unary_union(geoms).centroid)
)

def assign_clockwise_ids(group, county_centroid):
    cx, cy = county_centroid.x, county_centroid.y

    # Distance and angle of each polygon centroid relative to county centroid
    group = group.copy()
    group['_dx'] = group['centroid'].x - cx
    group['_dy'] = group['centroid'].y - cy
    group['_dist'] = np.hypot(group['_dx'], group['_dy'])
    group['_angle'] = np.arctan2(group['_dy'], group['_dx'])  # counter-clockwise radians

    # _001 = nearest to county centroid
    nearest_idx = group['_dist'].idxmin()
    start_angle = group.loc[nearest_idx, '_angle']

    # Clockwise: negate angles; shift so start_angle = 0; wrap to [0, 2π)
    group['_cw_angle'] = (-(group['_angle'] - start_angle)) % (2 * np.pi)

    # Nearest polygon gets 0 → sorts first
    group = group.sort_values('_cw_angle').reset_index(drop=True)

    # Assign Shape_ID
    county = group['County'].iloc[0]
    group['Shape_ID'] = [f"{county}_{i+1:03d}" for i in range(len(group))]

    return group.drop(columns=['_dx', '_dy', '_dist', '_angle', '_cw_angle'])

# Sort by county order first, then apply clockwise numbering per county
gdf_stage_2['County'] = pd.Categorical(gdf_stage_2['County'], categories=county_order, ordered=True)

result_parts = []
for county in county_order:
    group = gdf_stage_2[gdf_stage_2['County'] == county]
    if group.empty:
        continue
    numbered = assign_clockwise_ids(group, county_centroids[county])
    result_parts.append(numbered)

gdf_stage_2 = pd.concat(result_parts, ignore_index=True)

gdf_stage_2[['County', 'Shape_ID', 'Area']]

,County,Shape_ID,Area
0,NC1,NC1_001,0.2819
1,NC1,NC1_002,0.4191
2,NC1,NC1_003,0.3371
3,NC1,NC1_004,0.4594
4,NC1,NC1_005,0.2069
...,...,...,...
1704,WL2,WL2_064,41.0465
1705,WL2,WL2_065,26.9803
1706,WL2,WL2_066,0.5852
1707,WL2,WL2_067,1.0565


In [ ]:
import re

print("=== Shape_ID Validation ===\n")
all_valid = True

for county in county_order:
    group = gdf_stage_2[gdf_stage_2['County'] == county]
    ids = group['Shape_ID'].tolist()
    n = len(ids)
    expected = [f"{county}_{i:03d}" for i in range(1, n + 1)]

    bad_format = [s for s in ids if not re.fullmatch(rf"{re.escape(county)}_\d{{3}}", s)]
    missing = set(expected) - set(ids)
    duplicates = [s for s in ids if ids.count(s) > 1]

    if bad_format or missing or duplicates:
        all_valid = False
        print(f"[FAIL] {county} ({n} polygons)")
        if bad_format:   print(f"  Bad format:  {bad_format}")
        if missing:      print(f"  Missing IDs: {sorted(missing)}")
        if duplicates:   print(f"  Duplicates:  {list(set(duplicates))}")
    else:
        print(f"[ OK ] {county} — {county}_001 to {county}_{n:03d}")

print("\nAll Shape_IDs valid." if all_valid else "\nIssues found — review above.")

=== Shape_ID Validation ===

[ OK ] NC1 — NC1_001 to NC1_060
[ OK ] NC2 — NC2_001 to NC2_070
[ OK ] NC3 — NC3_001 to NC3_059
[ OK ] NC4 — NC4_001 to NC4_071
[ OK ] NC5 — NC5_001 to NC5_056
[ OK ] NC6 — NC6_001 to NC6_041
[ OK ] NC7 — NC7_001 to NC7_013
[ OK ] NC8 — NC8_001 to NC8_043
[ OK ] NC9 — NC9_001 to NC9_015
[ OK ] NC10 — NC10_001 to NC10_035
[ OK ] NC11 — NC11_001 to NC11_018
[ OK ] E1 — E1_001 to E1_052
[ OK ] E2 — E2_001 to E2_072
[ OK ] E3 — E3_001 to E3_037
[ OK ] E4 — E4_001 to E4_023
[ OK ] E5 — E5_001 to E5_017
[ OK ] E6 — E6_001 to E6_029
[ OK ] E7 — E7_001 to E7_038
[ OK ] E8 — E8_001 to E8_020
[ OK ] E9 — E9_001 to E9_107
[ OK ] E10 — E10_001 to E10_044
[ OK ] E11 — E11_001 to E11_091
[ OK ] E12 — E12_001 to E12_099
[ OK ] E13 — E13_001 to E13_053
[ OK ] E14 — E14_001 to E14_038
[ OK ] E15 — E15_001 to E15_039
[ OK ] E16 — E16_001 to E16_046
[ OK ] E17 — E17_001 to E17_031
[ OK ] E18 — E18_001 to E18_048
[ OK ] E19 — E19_001 to E19_063
[ OK ] W1 — W1_001 to W1_016
[ O

In [79]:
gdf_stage_2.to_csv('csv_input/stage3_synced_polygons.csv', index=False)